In [ ]:
import torch
import torch.nn as nn
import numpy as np
import tensorflow as tf

In [ ]:
class ModelConverter:
    @staticmethod
    def pytorch_to_onnx(model, input_shape, output_path='bird_classifier.onnx'):
        model.eval()
        dummy_input = torch.randn(1, *input_shape)
        
        torch.onnx.export(
            model,
            dummy_input,
            output_path,
            export_params=True,
            opset_version=11,
            do_constant_folding=True,
            input_names=['input'],
            output_names=['family_output', 'genus_output', 'species_output'],
            dynamic_axes={
                'input': {0: 'batch_size'},
                'family_output': {0: 'batch_size'},
                'genus_output': {0: 'batch_size'},
                'species_output': {0: 'batch_size'}
            }
        )
        print(f"Model exported to ONNX: {output_path}")
    
    @staticmethod
    def pytorch_to_tflite(model, input_shape, output_path='bird_classifier.tflite'):
        import onnx
        from onnx_tf.backend import prepare
        
        onnx_path = 'temp_model.onnx'
        ModelConverter.pytorch_to_onnx(model, input_shape, onnx_path)
        
        onnx_model = onnx.load(onnx_path)
        tf_rep = prepare(onnx_model)
        tf_rep.export_graph('temp_saved_model')
        
        converter = tf.lite.TFLiteConverter.from_saved_model('temp_saved_model')
        
        # Optimization for MCU
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        
        # Representative dataset for quantization
        def representative_dataset():
            for _ in range(100):
                data = np.random.randn(1, *input_shape).astype(np.float32)
                yield [data]
        
        converter.representative_dataset = representative_dataset
        
        tflite_model = converter.convert()
        
        with open(output_path, 'wb') as f:
            f.write(tflite_model)
        
        print(f"Model converted to TFLite: {output_path}")
        print(f"Model size: {len(tflite_model) / 1024:.2f} KB")
        
        return tflite_model


# Quantization-aware training
class QuantizedHierarchicalBirdClassifier(nn.Module):
    def __init__(self, num_families=5, num_genera=20, num_species=200, input_channels=1):
        super().__init__()
        
        self.quant = torch.quantization.QuantStub()
        self.dequant = torch.quantization.DeQuantStub()
        
        self.conv1 = nn.Conv2d(input_channels, 16, 3, 2, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu1 = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(16, 32, 3, 2, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(32)
        self.relu2 = nn.ReLU(inplace=True)
        
        self.conv3 = nn.Conv2d(32, 48, 3, 2, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(48)
        self.relu3 = nn.ReLU(inplace=True)
        
        self.conv4 = nn.Conv2d(48, 64, 3, 2, 1, bias=False)
        self.bn4 = nn.BatchNorm2d(64)
        self.relu4 = nn.ReLU(inplace=True)
        
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        
        self.family_classifier = nn.Linear(64, num_families)
        self.genus_classifier = nn.Linear(64, num_genera)
        self.species_classifier = nn.Linear(64, num_species)
        
    def forward(self, x):
        x = self.quant(x)
        
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.relu3(self.bn3(self.conv3(x)))
        x = self.relu4(self.bn4(self.conv4(x)))
        
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        
        family_logits = self.family_classifier(x)
        genus_logits = self.genus_classifier(x)
        species_logits = self.species_classifier(x)
        
        family_logits = self.dequant(family_logits)
        genus_logits = self.dequant(genus_logits)
        species_logits = self.dequant(species_logits)
        
        return family_logits, genus_logits, species_logits
    
    def fuse_model(self):
        torch.quantization.fuse_modules(self, [['conv1', 'bn1', 'relu1']], inplace=True)
        torch.quantization.fuse_modules(self, [['conv2', 'bn2', 'relu2']], inplace=True)
        torch.quantization.fuse_modules(self, [['conv3', 'bn3', 'relu3']], inplace=True)
        torch.quantization.fuse_modules(self, [['conv4', 'bn4', 'relu4']], inplace=True)

In [ ]:
def prepare_model_for_quantization(model):
    model.fuse_model()
    model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
    torch.quantization.prepare_qat(model, inplace=True)
    return model


def quantize_model(model):
    model.eval()
    quantized_model = torch.quantization.convert(model, inplace=False)
    return quantized_model


# TFLite inference wrapper for testing
class TFLiteInference:
    def __init__(self, model_path):
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()
        
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()
        
        print("Model loaded successfully!")
        print(f"Input shape: {self.input_details[0]['shape']}")
        print(f"Input type: {self.input_details[0]['dtype']}")
    
    def preprocess(self, mel_spec):
        # Normalize and quantize input
        mel_spec = mel_spec.astype(np.float32)
        
        # Get quantization parameters
        input_scale, input_zero_point = self.input_details[0]['quantization']
        
        if input_scale > 0:  # Model is quantized
            mel_spec = mel_spec / input_scale + input_zero_point
            mel_spec = np.clip(mel_spec, -128, 127).astype(np.int8)
        
        return mel_spec
    
    def predict(self, mel_spec):
        # Preprocess input
        input_data = self.preprocess(mel_spec)
        
        # Ensure correct shape
        if len(input_data.shape) == 3:
            input_data = np.expand_dims(input_data, axis=0)
        
        # Set input tensor
        self.interpreter.set_tensor(self.input_details[0]['index'], input_data)
        
        # Run inference
        self.interpreter.invoke()
        
        # Get outputs
        family_output = self.interpreter.get_tensor(self.output_details[0]['index'])
        genus_output = self.interpreter.get_tensor(self.output_details[1]['index'])
        species_output = self.interpreter.get_tensor(self.output_details[2]['index'])
        
        # Dequantize outputs if needed
        for i, output in enumerate([family_output, genus_output, species_output]):
            scale, zero_point = self.output_details[i]['quantization']
            if scale > 0:
                if i == 0:
                    family_output = (family_output.astype(np.float32) - zero_point) * scale
                elif i == 1:
                    genus_output = (genus_output.astype(np.float32) - zero_point) * scale
                else:
                    species_output = (species_output.astype(np.float32) - zero_point) * scale
        
        # Get predictions
        family_pred = np.argmax(family_output[0])
        genus_pred = np.argmax(genus_output[0])
        species_pred = np.argmax(species_output[0])
        
        return family_pred, genus_pred, species_pred


# C header file generator for MCU deployment
def generate_c_header(tflite_model_path, output_header_path='model_data.h'):
    with open(tflite_model_path, 'rb') as f:
        model_data = f.read()
    
    hex_array = ', '.join([f'0x{b:02x}' for b in model_data])
    
    header_content = f"""#ifndef MODEL_DATA_H
#define MODEL_DATA_H

const unsigned char model_data[] = {{
{hex_array}
}};

const unsigned int model_data_len = {len(model_data)};

#endif  // MODEL_DATA_H
"""
    
    with open(output_header_path, 'w') as f:
        f.write(header_content)
    
    print(f"C header file generated: {output_header_path}")
    print(f"Model size: {len(model_data) / 1024:.2f} KB")


# Complete deployment pipeline
def deploy_model_for_mcu(model, input_shape=(1, 64, 126)):
    print("Starting MCU deployment pipeline...")
    
    # Step 1: Convert to TFLite
    print("\n1. Converting model to TFLite format...")
    converter = ModelConverter()
    tflite_model = converter.pytorch_to_tflite(model, input_shape, 'bird_classifier.tflite')
    
    # Step 2: Generate C header
    print("\n2. Generating C header file...")
    generate_c_header('bird_classifier.tflite', 'model_data.h')
    
    # Step 3: Test inference
    print("\n3. Testing TFLite inference...")
    inference = TFLiteInference('bird_classifier.tflite')
    
    # Test with random input
    test_input = np.random.randn(1, 64, 126).astype(np.float32)
    family_pred, genus_pred, species_pred = inference.predict(test_input)
    
    print(f"Test prediction - Family: {family_pred}, Genus: {genus_pred}, Species: {species_pred}")
    
    print("\n4. Deployment files ready:")
    print("   - bird_classifier.tflite (for mobile/edge devices)")
    print("   - model_data.h (for MCU - include in Arduino/ESP32 project)")
    
    return inference

Label to index mapping:
dict_keys(['Abyssinian Thrush', 'African Bare-eyed Thrush', 'African Black-headed Oriole', 'African Blue Flycatcher', 'African Darter', 'African Dusky Flycatcher', 'African Emerald Cuckoo', 'African Fish-Eagle', 'African Goshawk', 'African Gray Flycatcher', 'African Gray Hornbill', 'African Green-Pigeon', 'African Jacana', 'African Paradise-Flycatcher', 'African Pied Wagtail', 'African Pygmy Kingfisher', 'African Sacred Ibis', 'African Thrush', 'Amethyst Sunbird', 'Augur Buzzard', 'Baglafecht Weaver', 'Barn Swallow', 'Beautiful Sunbird', 'Black Crake', 'Black Cuckoo', 'Black Kite', 'Black Sawwing', 'Black-and-white Mannikin', 'Black-and-white-casqued Hornbill', 'Black-backed Puffback', 'Black-collared Apalis', 'Black-crowned Tchagra', 'Black-faced Rufous-Warbler', 'Black-fronted Bushshrike', 'Black-headed Gonolek', 'Black-headed Heron', 'Black-necked Weaver', 'Black-tailed Oriole', 'Black-throated Apalis', 'Black-throated Barbet', 'Black-winged Lapwing', 'Blacks

In [6]:
with open("values.txt", 'a') as f:
    for key in label_to_idx.keys():
        text = f'"{key}",'
        f.write(text)
        # f.write(", ")

In [4]:
from torch.utils.data import DataLoader
from torchvision import transforms

# Step 2: Create dataset and dataloader
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # for CNN input
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = BirdSpectrogramDataset(config['train_images'], transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

test_dataset = BirdSpectrogramDataset(config['valid_images'], transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=True)

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from torchvision.models import efficientnet_b0
from torchvision.models.resnet import resnet18

class BirdClefModel(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True):
        super().__init__()
        self.num_classes = num_classes

        self.backbone = None
        self.in_features = None

        # Replace classifier head depending on model type
        if 'resnet' in model_name:
            self.backbone = resnet18(num_classes=num_classes)
            self.in_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Linear(self.in_features, num_classes)
            
            self.backbone.bn1 = nn.BatchNorm2d(64)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        elif 'efficientnet' in model_name:
            self.backbone = efficientnet_b0(num_classes=num_classes)
            self.in_features = 1000

            self.backbone.features[0][0] = nn.Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        else:
            raise ValueError(f"Unsupported model type for: {model_name}")
        
        if self.in_features is None:    
            raise ValueError("In Features cannot be None")

    def forward(self, x):
        x = self.backbone(x)
        return x

In [9]:
model = BirdClefModel('efficientnet', config['num_classes'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = model.to(device)

In [ ]:
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
  get_symmetric_quantization_config,
  XNNPACKQuantizer,
)
from torchao.quantization.pt2e.quantize_pt2e import (
  prepare_qat_pt2e,
  convert_pt2e,
  prepare_pt2e
)
import torchao

torch.save(model.state_dict(), 'orig_model.pth')

model.eval()

quantizer = XNNPACKQuantizer()

sample_inputs = (torch.randn(2, 1, 224, 224).to(device),)

exported_model = torch.export.export( 
    model, 
    sample_inputs).module()

quantizer.set_global(get_symmetric_quantization_config(is_qat=True))

model = prepare_qat_pt2e(exported_model, quantizer)

torchao.quantization.pt2e.move_exported_model_to_eval(model)

quantizer.set_global(get_symmetric_quantization_config())
model = prepare_pt2e(exported_model, quantizer)
quantized = convert_pt2e(model)

torch.save(quantized.state_dict(), 'best_model_qauantized.pth')

In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()), 
            lr=config['LR'],
            weight_decay=config['weight_decay']
        )

lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.2)

birds = list(label_to_idx.keys())

In [12]:
train_acc_history = []
val_acc_history = []
train_losses = []
val_losses = []

val_df = pd.DataFrame(columns=birds)
pred_df = pd.DataFrame(columns=birds)

for epoch in range(config['epochs']):
    # =========================== TRAINING =========================== #
    model.train()
    total_train_loss, total_train_acc, n_train_batches = 0.0, 0.0, 0
    print(f'Epoch number: {epoch}')

    for images, targets in tqdm(train_dataloader, desc="Training"):
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()

        logits = model(images)

        # print("TRAIN")
        # print(logits)

        loss = criterion(logits, targets)

        loss.backward()
        optimizer.step()

        # accuracy
        preds = torch.argmax(logits, dim=1)
        acc = (preds == targets).float().mean()

        total_train_loss += loss.item()
        total_train_acc += acc.item()
        n_train_batches += 1
        

    avg_loss = total_train_loss / n_train_batches
    avg_acc = total_train_acc / n_train_batches
    train_acc_history.append(avg_acc)
    train_losses.append(avg_loss)

    print(f"Train Loss: {avg_loss:.4f} | Train Acc: {avg_acc:.4f}")

    # =========================== VALIDATION =========================== #
    model.eval()
    
    total_val_loss, total_val_acc, n_val_batches = 0.0, 0.0, 0
    all_logits, all_targets = [], []

    with torch.no_grad():
        for images, targets in tqdm(test_dataloader, desc="Validating"):
            images, targets = images.to(device), targets.to(device)
            logits = model(images)

            # print("VALIDATION")
            # print(logits)
                
            loss = criterion(logits, targets)

            preds = torch.argmax(logits, dim=1)
            acc = (preds == targets).float().mean()

            total_val_loss += loss.item()
            total_val_acc += acc.item()
            n_val_batches += 1

        avg_loss = total_val_loss / n_val_batches
        avg_acc = total_val_acc / n_val_batches
        val_acc_history.append(avg_acc)
        val_losses.append(avg_loss)

        print(f"Val Loss: {avg_loss:.4f} | Val Acc: {avg_acc:.4f}")

Epoch number: 0


Training: 100%|██████████| 841/841 [02:46<00:00,  5.07it/s]


Train Loss: 4.6989 | Train Acc: 0.0682


Validating: 100%|██████████| 219/219 [00:31<00:00,  6.89it/s]

Val Loss: 4.2134 | Val Acc: 0.1327


In [ ]:
torch.save(model.state_dict(), 'orig_model.pth')

In [ ]:
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
    get_symmetric_quantization_config,
    XNNPACKQuantizer,
)

from torchao.quantization.pt2e.quantize_pt2e import (
  prepare_qat_pt2e,
)

example_inputs = (torch.rand(1, 1, 224, 224),)
float_model = BirdClefModel("efficientnet", 264)
float_model.eval()

exported_model = torch.export.export(float_model, example_inputs).module()

quantizer = XNNPACKQuantizer()
quantizer.set_global(get_symmetric_quantization_config(is_qat=True))

prepared_model = prepare_qat_pt2e(exported_model, quantizer)
prepared_model.load_state_dict(torch.load("best_model_qat1.pth"))

INFO:tensorflow:Assets written to: /tmp/tmpmntitsrb/assets


INFO:tensorflow:Assets written to: /tmp/tmpmntitsrb/assets
W0000 00:00:1760797964.924407    8075 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1760797964.924427    8075 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2025-10-18 20:32:44.924690: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpmntitsrb
2025-10-18 20:32:44.928897: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-10-18 20:32:44.928906: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpmntitsrb
I0000 00:00:1760797964.963965    8075 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2025-10-18 20:32:44.969347: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-10-18 20:32:45.250749: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpmntitsrb
2025-10-18 20:32:45.317

In [18]:
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training & Validation Losses')
plt.legend()
plt.savefig('train_val_loss_plot.png')
plt.close()

In [19]:
plt.figure(figsize=(10, 6))
plt.plot(train_acc_history, label='Training Accuracy')
plt.plot(val_acc_history, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training & Validation Accuracy')
plt.ylim(0, 1)
plt.legend()
plt.savefig('accuracy_plot.png')
plt.close()

In [ ]:
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

from torchao.quantization.pt2e.quantize_pt2e import (
  convert_pt2e,
)

quantized_model = convert_pt2e(prepared_model)

sample_inputs = (torch.randn(2, 1, 224, 224),)

et_program = to_edge_transform_and_lower( # (6)
    torch.export.export(quantized_model, sample_inputs),
    partitioner=[XnnpackPartitioner()],
).to_executorch()

# 3. Save for deployment
with open("model.pte", "wb") as f:
    f.write(et_program.buffer)

/home/mushahid/anaconda3/envs/esp32-env/lib/python3.11/site-packages/torch/fx/graph.py:1157: UserWarning: erase_node(batch_norm_98) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/mushahid/anaconda3/envs/esp32-env/lib/python3.11/site-packages/torch/fx/graph.py:1157: UserWarning: erase_node(batch_norm_99) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/mushahid/anaconda3/envs/esp32-env/lib/python3.11/site-packages/torch/fx/graph.py:1157: UserWarning: erase_node(batch_norm_100) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/mushahid/anaconda3/envs/esp32-env/lib/python3.11/site-packages/torch/fx/graph.py:1157: UserWarning: erase_node(batch_norm_101) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/mushahid/anaconda3/envs/esp32-env/lib/python3.11/site-packages/torch/fx/graph.py:115